# Task 6: Bidirectional Long Short-Term Memory (BiLSTM) Cell Mechanics from Scratch

## Objective

Implement a Bidirectional LSTM layer from scratch using PyTorch tensor operations and NumPy.

No `nn.RNN`, `nn.LSTM`, or other recurrent-layer implementations are used.

The implementation explicitly calculates:

- Input gate \(i_t\)
- Forget gate \(f_t\)
- Output gate \(o_t\)
- Cell candidate \(g_t\)
- Cell state \(c_t\)
- Hidden state \(h_t\)
- Forward sequence processing
- Backward sequence processing
- Concatenation of both directions
- Variable-length padded sequences

In [ ]:
import torch
import numpy as np

torch.set_grad_enabled(False)

torch.manual_seed(42)
np.random.seed(42)

# ------------------------------------------------------------
# Variable-length padded sequence batch
# ------------------------------------------------------------

batch_size = 4
max_seq_len = 7
input_size = 5
hidden_size = 8

lengths = torch.tensor([
    7, 5, 4, 6
])

X = torch.randn(
    batch_size,
    max_seq_len,
    input_size
)

# Zero padding after each sequence length
for i, length in enumerate(lengths):

    X[i, length:] = 0

print("Input shape:", X.shape)
print("Sequence lengths:", lengths.tolist())

# LSTM Parameters and Single-Step Cell

All four gate transformations are implemented manually.

The gates are combined into one matrix multiplication for efficiency:

\[
z_t =
x_tW_x+h_{t-1}W_h+b
\]

The resulting vector is split into four sections corresponding to:

\[
[i_t,f_t,g_t,o_t]
\]

No PyTorch recurrent modules are used.

In [ ]:
def sigmoid(x):
    return 1 / (1 + torch.exp(-x))


def create_lstm_parameters(
    input_size,
    hidden_size
):

    # Input-to-hidden weights
    W_x = torch.randn(
        input_size,
        4 * hidden_size
    ) * 0.1

    # Hidden-to-hidden weights
    W_h = torch.randn(
        hidden_size,
        4 * hidden_size
    ) * 0.1

    bias = torch.zeros(
        4 * hidden_size
    )

    return W_x, W_h, bias


def lstm_step(
    x_t,
    h_prev,
    c_prev,
    W_x,
    W_h,
    bias
):

    z = (
        x_t @ W_x
        + h_prev @ W_h
        + bias
    )

    # Split into four gates
    i, f, g, o = torch.chunk(
        z,
        4,
        dim=1
    )

    i = sigmoid(i)
    f = sigmoid(f)
    g = torch.tanh(g)
    o = sigmoid(o)

    # Cell state
    c = (
        f * c_prev
        + i * g
    )

    # Hidden state
    h = (
        o
        * torch.tanh(c)
    )

    return h, c


# Create separate parameters
# for forward and backward directions
forward_params = create_lstm_parameters(
    input_size,
    hidden_size
)

backward_params = create_lstm_parameters(
    input_size,
    hidden_size
)

print("Forward W_x:", forward_params[0].shape)
print("Forward W_h:", forward_params[1].shape)

# Forward and Backward Sequence Processing

The forward direction processes:

\[
t=0,1,\ldots,T-1
\]

The backward direction processes:

\[
t=T-1,T-2,\ldots,0
\]

Variable-length sequences are handled using the supplied `lengths` tensor.

Once a sequence reaches its actual length, its hidden and cell states are kept unchanged for padded positions.

Finally:

\[
H_t =
[H_t^{forward};H_t^{backward}]
\]

produces the bidirectional representation.

In [ ]:
def run_direction(
    X,
    lengths,
    params,
    reverse=False
):

    batch_size, max_len, _ = X.shape

    W_x, W_h, bias = params

    h = torch.zeros(
        batch_size,
        hidden_size
    )

    c = torch.zeros(
        batch_size,
        hidden_size
    )

    outputs = torch.zeros(
        batch_size,
        max_len,
        hidden_size
    )

    # Forward or backward temporal order
    if reverse:
        time_steps = range(
            max_len - 1,
            -1,
            -1
        )
    else:
        time_steps = range(
            max_len
        )

    for t in time_steps:

        # Valid positions for this timestep
        if reverse:

            valid = lengths > t

        else:

            valid = lengths > t

        x_t = X[:, t, :]

        h_new, c_new = lstm_step(
            x_t,
            h,
            c,
            W_x,
            W_h,
            bias
        )

        # Keep states unchanged for padded positions
        mask = valid.float().unsqueeze(1)

        h = (
            mask * h_new
            + (1 - mask) * h
        )

        c = (
            mask * c_new
            + (1 - mask) * c
        )

        outputs[:, t, :] = (
            mask * h
        )

    return outputs


def bidirectional_lstm(
    X,
    lengths,
    forward_params,
    backward_params
):

    forward_output = run_direction(
        X,
        lengths,
        forward_params,
        reverse=False
    )

    backward_output = run_direction(
        X,
        lengths,
        backward_params,
        reverse=True
    )

    # Concatenate both directions
    output = torch.cat(
        [
            forward_output,
            backward_output
        ],
        dim=2
    )

    return output


output = bidirectional_lstm(
    X,
    lengths,
    forward_params,
    backward_params
)

print("BiLSTM output shape:", output.shape)

In [ ]:
# ============================================================
# Verify variable-length handling
# ============================================================

print("Expected output shape:")
print(
    f"({batch_size}, {max_seq_len}, "
    f"{2 * hidden_size})"
)

print("\nActual output shape:")
print(output.shape)

print("\nValid output norms:")

for i, length in enumerate(lengths):

    valid_output = output[
        i,
        :length
    ]

    print(
        f"Sequence {i+1}: "
        f"length={length.item()}, "
        f"norm={torch.norm(valid_output).item():.4f}"
    )

print("\nPadded output norms:")

for i, length in enumerate(lengths):

    padded_output = output[
        i,
        length:
    ]

    if padded_output.numel() == 0:

        print(
            f"Sequence {i+1}: no padding"
        )

    else:

        print(
            f"Sequence {i+1}: "
            f"norm={torch.norm(padded_output).item():.4f}"
        )

# Conclusion

A Bidirectional LSTM cell was successfully implemented from scratch using PyTorch tensor operations.

The implementation explicitly calculated the four LSTM gates:

\[
i_t,\quad f_t,\quad g_t,\quad o_t
\]

and updated the cell and hidden states using:

\[
c_t=f_t\odot c_{t-1}+i_t\odot g_t
\]

\[
h_t=o_t\odot\tanh(c_t)
\]

Two independent LSTM directions were created.

The forward direction processed the sequence from left to right, while the backward direction processed it from right to left.

Their hidden representations were concatenated to produce the final Bidirectional LSTM output:

\[
H_t =
[H_t^{forward};H_t^{backward}]
\]

Variable-length padded sequences were also handled using explicit sequence lengths, preventing padded timesteps from changing the recurrent states.

Therefore, the experiment demonstrates the internal mechanics of a BiLSTM without relying on `nn.RNN`, `nn.LSTM`, or other pre-built recurrent layers.